In [ ]:

!pip uninstall -y transformers adapters huggingface_hub accelerate peft sentence-transformers
!pip install -q huggingface_hub==0.23.0 transformers==4.39.3 adapters==0.2.1 accelerate==0.27.2 datasets kaggle

import os, shutil, json, torch, nltk, numpy as np
from google.colab import drive
from transformers import AutoTokenizer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
from adapters import AutoAdapterModel, AdapterTrainer

os.environ["WANDB_DISABLED"] = "true"

try:
    torch.serialization.add_safe_globals([np.core.multiarray._reconstruct, np.ndarray, np.dtype])
except AttributeError: pass

if not hasattr(torch, 'original_load_func'): torch.original_load_func = torch.load
def safe_load_override(*args, **kwargs):
    if 'weights_only' in kwargs: del kwargs['weights_only']
    return torch.original_load_func(*args, weights_only=False, **kwargs)
torch.load = safe_load_override

os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY'] = "0ba946628cb1f5acb76ecd357f590e95"
drive.mount('/content/drive')

FINAL_SAVE_PATH = "/content/drive/My Drive/SLM_Research/T5_Small_Spider_Houlsby"
CHECKPOINT_DIR = "/content/drive/My Drive/SLM_Research/T5_Small_Spider_Houlsby/checkpoints"

if not os.path.exists('spider_data'):
    !kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
    !unzip -q yale-universitys-spider-10-nlp-dataset.zip -d temp_spider
    shutil.move("temp_spider/spider", "spider_data")
    shutil.rmtree('temp_spider')
    !wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py
    !wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py
    nltk.download('punkt')
    nltk.download('punkt_tab')

with open('spider_data/tables.json', 'r') as f: table_data = json.load(f)
schema_map = {db['db_id']: db for db in table_data}

def get_crp_schema(db_id):
    if db_id not in schema_map: return ""
    db = schema_map[db_id]
    ddls = []
    for i, table_name in enumerate(db['table_names_original']):
        cols = [f"{c[1]} {db['column_types'][j]}" for j, c in enumerate(db['column_names_original']) if c[0] == i]
        ddls.append(f"CREATE TABLE {table_name} ({', '.join(cols)});")
    return " ".join(ddls)

MODEL_NAME = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_fn(examples):
    inputs = [f"translate to SQL: {q} | Schema: {get_crp_schema(d)}" for q, d in zip(examples['question'], examples['db_id'])]
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        padding="max_length",
        truncation=True,
        truncation_strategy="longest_first"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples['query'], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def load_ds(path):
    with open(path, 'r', encoding='utf-8') as f: data = json.load(f)
    return Dataset.from_dict({"question": [x["question"] for x in data], "query": [x["query"] for x in data], "db_id": [x["db_id"] for x in data]})

train_ds = load_ds("spider_data/train_spider.json").map(preprocess_fn, batched=True)
val_ds = load_ds("spider_data/dev.json").map(preprocess_fn, batched=True)

model = AutoAdapterModel.from_pretrained(MODEL_NAME)
adapter_name = "spider_t5"

model.add_adapter(adapter_name, config="seq_bn")
model.add_seq2seq_lm_head(adapter_name)
model.set_active_adapters(adapter_name)
model.train_adapter(adapter_name)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    weight_decay=0.01,
    fp16=False,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    report_to="none",
    logging_steps=50,
    load_best_model_at_end=False
)

trainer = AdapterTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator
)

trainer.train(resume_from_checkpoint=True)


model.save_adapter(FINAL_SAVE_PATH, adapter_name)
tokenizer.save_pretrained(FINAL_SAVE_PATH)

model.eval()
predictions, gold_lines = [], []
for item in val_ds:
    input_text = f"translate to SQL: {item['question']} | Schema: {get_crp_schema(item['db_id'])}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    predictions.append(tokenizer.decode(output[0], skip_special_tokens=True) + "\n")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

with open('evaluation.py', 'r', encoding='utf-8') as f:
    eval_code = f.read()

eval_code = eval_code.replace(
    "conn = sqlite3.connect(db)",
    "conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors='ignore')"
)

with open('evaluation.py', 'w', encoding='utf-8') as f:
    f.write(eval_code)

!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: huggingface_hub 1.5.0
Uninstalling huggingface_hub-1.5.0:
  Successfully uninstalled huggingface_hub-1.5.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warning

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss


Token indices sequence length is longer than the specified maximum sequence length for this model (517 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (517 > 512). Running this sequence through the model will result in indexing errors


Streaming output truncated to the last 5000 lines.
extra gold: SELECT mpg FROM CARS_DATA WHERE Cylinders  =  8 OR YEAR  <  1980 ORDER BY mpg DESC LIMIT 1;

extra pred: SELECT max(mpg) FROM cars_data GROUP BY max(mpg) DESC LIMIT 1
extra gold: SELECT mpg FROM CARS_DATA WHERE Cylinders  =  8 OR YEAR  <  1980 ORDER BY mpg DESC LIMIT 1;

eval_err_num:103
extra pred: SELECT Model FROM car_names AS T1 JOIN model_list AS T2 ON T1.Model = T2.Model WHERE T1.Continent > 3500 AND T1.Model = 'Ford Motor Company'
extra gold: SELECT DISTINCT T1.model FROM MODEL_LIST AS T1 JOIN CAR_NAMES AS T2 ON T1.Model  =  T2.Model JOIN CARS_DATA AS T3 ON T2.MakeId  =  T3.Id JOIN CAR_MAKERS AS T4 ON T1.Maker  =  T4.Id WHERE T3.weight  <  3500 AND T4.FullName != 'Ford Motor Company';

eval_err_num:104
extra pred: SELECT DISTINCT Model FROM model_list AS T1 JOIN cars_data AS T2 ON T1.Model = T2.Model WHERE T1.MPG > 3500 AND T2.Model = T2.Model WHERE T2.MPG > 3500 AND T2.Model = "Fort Motor Company"
extra gold: SELECT

In [1]:
!pip uninstall -y transformers adapters huggingface_hub accelerate peft sentence-transformers
!pip install -q huggingface_hub==0.23.0 transformers==4.39.3 adapters==0.2.1 accelerate==0.27.2 datasets kaggle

# ĐÁNH GIÁ HIỆU NĂNG CHO MÔ HÌNH ADAPTERS (T5-Small)

import torch
import time
import numpy as np
import psutil
import os
import gc
from transformers import AutoTokenizer
from adapters import AutoAdapterModel
from google.colab import drive

# Kết nối Drive
drive.mount('/content/drive')

# Dọn dẹp bộ nhớ
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

BASE_MODEL_NAME = "t5-small"
ADAPTER_SAVE_PATH = "/content/drive/My Drive/SLM_Research/T5_Small_Spider_Houlsby"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Tải Tokenizer và Base Model
print("Đang tải mô hình gốc và cấu hình Adapters...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_SAVE_PATH)
model = AutoAdapterModel.from_pretrained(BASE_MODEL_NAME)

# Kích hoạt Adapter
model.load_adapter(ADAPTER_SAVE_PATH, set_active=True)
model.to(device)
model.eval()
print("✅ Tải thành công mô hình với Adapters!")

# Dữ liệu đầu vào
input_text = "translate to SQL: How many students are there? | Schema: CREATE TABLE student(id int, name text);"
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)

# Warmup GPU
print("Đang warmup GPU...")
for _ in range(10):
    with torch.no_grad():
        model.generate(**inputs, max_length=128)

# Đo Latency (Độ trễ)
print("Đang đo lường độ trễ (100 lần chạy)...")
latencies = []
for _ in range(100):
    start = time.time()
    with torch.no_grad():
        model.generate(**inputs, max_length=128)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latencies.append(time.time() - start)

avg_latency = np.mean(latencies)
std_latency = np.std(latencies)

# Thông lượng (Throughput)
throughput = 1 / avg_latency

# Đo VRAM và RAM
gpu_memory = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else None
process = psutil.Process(os.getpid())
ram_usage = process.memory_info().rss / 1024**2
total_params = sum(p.numel() for p in model.parameters())

# In kết quả
print("\n" + "="*45)
print("   ĐÁNH GIÁ HIỆU NĂNG (T5-SMALL + ADAPTERS)   ")
print("="*45)
print(f"Độ trễ TB (Latency):            {avg_latency*1000:.2f} ms")
print(f"Độ lệch chuẩn (Std):            {std_latency*1000:.2f} ms")
print(f"Thông lượng (Throughput BS=1):  {throughput:.2f} samples/sec")
if gpu_memory:
    print(f"VRAM tối đa đã dùng (Peak):     {gpu_memory:.2f} MB")
print(f"RAM CPU đang sử dụng:           {ram_usage:.2f} MB")
print(f"Tổng số tham số mô hình:        {total_params:,}")
print("="*45)

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: huggingface_hub 1.5.0
Uninstalling huggingface_hub-1.5.0:
  Successfully uninstalled huggingface_hub-1.5.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Tải thành công mô hình với Adapters!
Đang warmup GPU...
Đang đo lường độ trễ (100 lần chạy)...

   ĐÁNH GIÁ HIỆU NĂNG (T5-SMALL + ADAPTERS)   
Độ trễ TB (Latency):            234.95 ms
Độ lệch chuẩn (Std):            55.84 ms
Thông lượng (Throughput BS=1):  4.26 samples/sec
VRAM tối đa đã dùng (Peak):     245.95 MB
RAM CPU đang sử dụng:           1556.66 MB
Tổng số tham số mô hình:        60,906,368
